In [10]:
import sqlite3
import pandas as pd

from pathlib import Path

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = Path.cwd().parent
BUILTIN_PACKS_DIR = PROJECT_ROOT / "packs"
DB_PATH_GEONAMES = PROJECT_ROOT / "data" / "geonames.sqlite"

In [11]:
def run_query(query: str, params: tuple = ()) -> pd.DataFrame:
    conn = sqlite3.connect(DB_PATH_GEONAMES)
    try:
        return pd.read_sql_query(query, conn, params=params)
    finally:
        conn.close()


def table_exists(table_name: str) -> bool:
    df = run_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table' AND name = ?
        """,
        (table_name,),
    )
    return not df.empty

In [12]:
run_query("""
SELECT
    alternate_name,
    is_preferred_name,
    is_short_name,
    is_colloquial,
    is_historic,
    from_date,
    to_date,
    row_kind
FROM alternate_name
WHERE geonameid = 745044
ORDER BY
    is_preferred_name DESC,
    is_short_name DESC,
    is_historic ASC,
    alternate_name;
    """)

,alternate_name,is_preferred_name,is_short_name,is_colloquial,is_historic,from_date,to_date,row_kind
0,Estambul,1,0,0,0,None,None,name_lang
1,Istambul,1,0,0,0,None,None,name_lang
2,Istanbul,1,0,0,0,None,None,name_lang
3,Istanbul,1,0,0,0,None,None,name_lang
4,Istanbul,1,0,0,0,None,None,name_lang
5,Istanbul,1,0,0,0,None,None,name_lang
6,Istanbul,1,0,0,0,None,None,name_lang
7,Istanbul,1,0,0,0,None,None,name_lang
8,Istanbul,1,0,0,0,None,None,name_lang
9,Istanbul,1,0,0,0,None,None,name_lang


In [13]:
run_query("""
SELECT
    isolanguage,
    alternate_name,
    is_preferred_name,
    is_historic,
    from_date,
    to_date
FROM alternate_name
WHERE geonameid = 745044
  AND (
      isolanguage = 'la'
      OR lower(alternate_name) LIKE 'constantin%'
      OR lower(alternate_name) LIKE 'nova roma%'
      OR lower(alternate_name) LIKE 'byzant%'
  )
ORDER BY isolanguage, alternate_name;
""")

,isolanguage,alternate_name,is_preferred_name,is_historic,from_date,to_date
0,,Byzantion,0,1,None,None
1,,Constantinopoli,0,1,None,None
2,,Constantinopolis,0,1,None,None
3,ca,Constantinoble,0,1,None,None
4,cs,Byzantion,0,1,None,None
5,de,Byzantion,0,1,None,None
6,en,Byzantium,0,1,None,None
7,en,Constantinople,0,1,None,None
8,es,Constantinopla,0,1,None,None
9,fi,Byzantion,0,1,None,None


In [14]:
pack_tables = [
    "pack",
    "pack_build",
    "pack_fallback_step",
    "pack_place_override",
    "pack_exonym",
]

pd.DataFrame(
    [{"table": t, "exists": table_exists(t)} for t in pack_tables]
)

,table,exists
0,pack,True
1,pack_build,True
2,pack_fallback_step,True
3,pack_place_override,True
4,pack_exonym,True


In [15]:
rows = []
for table_name in pack_tables:
    if table_exists(table_name):
        count = run_query(f"SELECT COUNT(*) AS n FROM {table_name}")["n"].iloc[0]
        rows.append({"table": table_name, "rows": count})

pd.DataFrame(rows)

,table,rows
0,pack,1
1,pack_build,1
2,pack_fallback_step,3
3,pack_place_override,3
4,pack_exonym,4


In [16]:
run_query("""
SELECT *
FROM pack
ORDER BY pack_id
""")

,pack_id,display_name,origin,version,default_mode,enabled,source_path,manifest_json,content_hash
0,latin,Latin,builtin,0.1.0,lookup_first,1,/home/max/Documents/Programming/NameTransducti...,"{""bcp47"": ""la"", ""date_policy"": {""default_era"":...",7802e27ecb3d0d93c50fb2dc782c127db37cea1a5d53ba...


In [17]:
run_query("""
SELECT *
FROM pack_build
ORDER BY pack_id
""")

,pack_id,built_at,content_hash
0,latin,2026-04-08T16:40:43.887882+00:00,7802e27ecb3d0d93c50fb2dc782c127db37cea1a5d53ba...


In [18]:
run_query("""
SELECT *
FROM pack_fallback_step
WHERE pack_id = 'latin'
ORDER BY step_index
""")

,pack_id,step_index,step_type,step_target,config_json
0,latin,0,source_lookup,geonames,"{""source"": ""geonames"", ""type"": ""source_lookup""}"
1,latin,1,pack_exonym_dictionary,NaN,"{""type"": ""pack_exonym_dictionary""}"
2,latin,2,strict_transliteration,NaN,"{""type"": ""strict_transliteration""}"


In [19]:
run_query("""
SELECT *
FROM pack_place_override
WHERE pack_id = 'latin'
ORDER BY namespace, external_id, priority DESC, valid_from
""")

,pack_id,namespace,external_id,geonameid,output_name,output_name_normalized,valid_from,valid_to,priority,tags,note,source_file,source_line
0,latin,geonames,745044,745044,Byzantium,byzantium,-0660-01-01,0329-12-31,1000,"antique,historic",Preferred antiquity-era Latin form for Istanbu...,place_overrides.tsv,2
1,latin,geonames,745044,745044,Constantinopolis,constantinopolis,0330-01-01,1453-12-31,1000,"medieval,historic",Preferred late antique / medieval Latin form,place_overrides.tsv,3
2,latin,geonames,745044,745044,Nova Roma,nova roma,0330-01-01,0400-12-31,400,"ceremonial,historic",Ceremonial variant,place_overrides.tsv,4


In [20]:
run_query("""
SELECT *
FROM pack_exonym
WHERE pack_id = 'latin'
ORDER BY input_normalized, priority DESC
""")

,pack_id,input_normalized,output_name,output_name_normalized,valid_from,valid_to,priority,note,source_file,source_line
0,latin,constantinople,Constantinopolis,constantinopolis,0330-01-01,1453-12-31,250,Canonical Latinized form,exonyms.tsv,4
1,latin,istanbul,Constantinopolis,constantinopolis,0330-01-01,1453-12-31,200,Default medieval Latin output before Ottoman era,exonyms.tsv,2
2,latin,istanbul,Byzantium,byzantium,-0660-01-01,0329-12-31,200,Default antique Latin output,exonyms.tsv,3
3,latin,new york,Novum Eboracum,novum eboracum,1500-01-01,2100-12-31,50,Neo-Latin form,exonyms.tsv,5


In [21]:
run_query("""
SELECT
    pack_id,
    namespace,
    external_id,
    geonameid,
    output_name,
    valid_from,
    valid_to,
    priority,
    tags,
    note
FROM pack_place_override
WHERE pack_id = 'latin'
  AND geonameid = 745044
ORDER BY priority DESC, valid_from
""")

,pack_id,namespace,external_id,geonameid,output_name,valid_from,valid_to,priority,tags,note
0,latin,geonames,745044,745044,Byzantium,-0660-01-01,0329-12-31,1000,"antique,historic",Preferred antiquity-era Latin form for Istanbu...
1,latin,geonames,745044,745044,Constantinopolis,0330-01-01,1453-12-31,1000,"medieval,historic",Preferred late antique / medieval Latin form
2,latin,geonames,745044,745044,Nova Roma,0330-01-01,0400-12-31,400,"ceremonial,historic",Ceremonial variant


In [22]:
run_query("""
SELECT
    output_name,
    valid_from,
    valid_to,
    priority,
    tags
FROM pack_place_override
WHERE pack_id = 'latin'
  AND geonameid = 745044
  AND (valid_from IS NULL OR valid_from <= '0140-01-01')
  AND (valid_to   IS NULL OR valid_to   >= '0140-01-01')
ORDER BY priority DESC, valid_from DESC
LIMIT 1
""")

,output_name,valid_from,valid_to,priority,tags
0,Byzantium,-0660-01-01,0329-12-31,1000,"antique,historic"


In [23]:
run_query("""
SELECT
    output_name,
    valid_from,
    valid_to,
    priority,
    tags
FROM pack_place_override
WHERE pack_id = 'latin'
  AND geonameid = 745044
  AND (valid_from IS NULL OR valid_from <= '1100-01-01')
  AND (valid_to   IS NULL OR valid_to   >= '1100-01-01')
ORDER BY priority DESC, valid_from DESC
LIMIT 1
""")

,output_name,valid_from,valid_to,priority,tags
0,Constantinopolis,0330-01-01,1453-12-31,1000,"medieval,historic"


In [24]:
run_query("""
SELECT
    ppo.pack_id,
    ppo.output_name AS pack_output,
    ppo.valid_from,
    ppo.valid_to,
    ppo.priority,
    g.geonameid,
    g.name AS geonames_name,
    g.country_code,
    g.feature_code,
    g.population
FROM pack_place_override AS ppo
JOIN geoname AS g
  ON g.geonameid = ppo.geonameid
WHERE ppo.pack_id = 'latin'
ORDER BY g.population DESC, ppo.priority DESC
""")

,pack_id,pack_output,valid_from,valid_to,priority,geonameid,geonames_name,country_code,feature_code,population
0,latin,Byzantium,-0660-01-01,0329-12-31,1000,745044,Istanbul,TR,PPLA,15701602
1,latin,Constantinopolis,0330-01-01,1453-12-31,1000,745044,Istanbul,TR,PPLA,15701602
2,latin,Nova Roma,0330-01-01,0400-12-31,400,745044,Istanbul,TR,PPLA,15701602


In [25]:
run_query("""
SELECT
    pack_id,
    COUNT(*) AS override_count,
    COUNT(DISTINCT geonameid) AS distinct_places
FROM pack_place_override
GROUP BY pack_id
ORDER BY pack_id
""")

,pack_id,override_count,distinct_places
0,latin,3,1


In [26]:
def resolve_override(pack_id: str, geonameid: int, date: str) -> pd.DataFrame:
    return run_query(
        """
        SELECT
            output_name,
            valid_from,
            valid_to,
            priority,
            tags,
            note
        FROM pack_place_override
        WHERE pack_id = ?
          AND geonameid = ?
          AND (valid_from IS NULL OR valid_from <= ?)
          AND (valid_to   IS NULL OR valid_to   >= ?)
        ORDER BY priority DESC, valid_from DESC
        LIMIT 1
        """,
        (pack_id, geonameid, date, date),
    )

In [27]:
resolve_override("latin", 745044, "0140-01-01")

,output_name,valid_from,valid_to,priority,tags,note
0,Byzantium,-0660-01-01,0329-12-31,1000,"antique,historic",Preferred antiquity-era Latin form for Istanbu...


In [28]:
resolve_override("latin", 745044, "1100-01-01")

,output_name,valid_from,valid_to,priority,tags,note
0,Constantinopolis,0330-01-01,1453-12-31,1000,"medieval,historic",Preferred late antique / medieval Latin form
